In [6]:
# Install required packages quietly
!pip install wikipedia sentence-transformers faiss-cpu transformers torch --upgrade -q

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 81.0 MB/s eta 0:00:00


In [7]:
# Create the sample corpus directory
!mkdir -p sample_corpus

In [9]:
%%writefile sample_corpus/rag_explained.txt
Retrieval-Augmented Generation (RAG) is an AI framework that improves the quality of language model responses by grounding them in external sources of knowledge.
Instead of relying solely on its pre-trained data, a RAG system first retrieves relevant documents or passages from a knowledge base
(like a collection of text files or a database) and then uses this information as context to generate a more accurate and factual answer.

Writing sample_corpus/rag_explained.txt


In [10]:
%%writefile sample_corpus/llms_intro.txt
A Large Language Model (LLM) is a type of artificial intelligence model trained on vast amounts of text data to understand and generate human-like language.
Key examples include models from the GPT family, LLaMA, and PaLM. They can perform a wide range of tasks, such as translation, summarization, and question-answering,
by predicting the next word in a sequence.

Writing sample_corpus/llms_intro.txt


In [15]:
# Add this in a new cell and run it
%%writefile sample_corpus/solar_power.txt
Solar power is the conversion of energy from sunlight into electricity. It is a renewable energy source, and its technologies are broadly characterized
as either passive solar or active solar depending on how they capture and distribute solar energy or convert it into solar power.
Active solar techniques include the use of photovoltaic systems, concentrated solar power, and solar water heating to harness the energy.

Writing sample_corpus/solar_power.txt


In [21]:
import os
import json
import argparse
import time
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional

# ---- Tool 0: Utilities
def truncate(text: str, n: int = 1200) -> str:
    return text if len(text) <= n else text[:n] + "...[truncated]"

# ---- Tool 1: Local Corpus Retrieval (FAISS + sentence-transformers)
class LocalCorpusTool:
    def __init__(self, docs_dir: str, index_path: str, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.docs_dir = docs_dir
        self.index_path = index_path
        self.model_name = model_name
        self.emb = None
        self.faiss = None
        self.doc_texts = []
        self.doc_names = []

        if not os.path.isdir(docs_dir):
            os.makedirs(docs_dir, exist_ok=True)

    def _lazy_import(self):
        global faiss, SentenceTransformer
        import faiss  # type: ignore
        from sentence_transformers import SentenceTransformer  # type: ignore
        self.faiss = faiss
        if self.emb is None:
            self.emb = SentenceTransformer(self.model_name)

    def _load_docs(self):
        self.doc_texts, self.doc_names = [], []
        for fn in os.listdir(self.docs_dir):
            if fn.lower().endswith((".txt", ".md")):
                path = os.path.join(self.docs_dir, fn)
                with open(path, "r", encoding="utf-8") as f:
                    txt = f.read().strip()
                self.doc_texts.append(txt)
                self.doc_names.append(fn)

    def build_or_load(self):
        self._lazy_import()
        self._load_docs()
        if not self.doc_texts:
            print("Warning: No documents found in the local corpus directory.")
            self.index = None
            return
        vectors = self.emb.encode(self.doc_texts, show_progress_bar=False)
        d = vectors.shape[1]
        index = self.faiss.IndexFlatIP(d)  # cosine via normalized dot
        import numpy as np
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-10
        vectors = vectors / norms
        index.add(vectors.astype("float32"))
        self.index = index

    def search(self, query: str, k: int = 3) -> List[Dict[str, str]]:
        self._lazy_import()
        if not hasattr(self, "index"):
            self.build_or_load()
        if self.index is None:
            return []
        qv = self.emb.encode([query], show_progress_bar=False)
        import numpy as np
        qv = qv / (np.linalg.norm(qv, axis=1, keepdims=True) + 1e-10)
        D, I = self.index.search(qv.astype("float32"), k)
        results = []
        for idx in I[0]:
            if idx < 0 or idx >= len(self.doc_texts):
                continue
            results.append({"source": f"local:{self.doc_names[idx]}", "text": self.doc_texts[idx]})
        return results

# ---- Tool 2: Wikipedia Search (online)
class WikipediaTool:
    def __init__(self, lang: str = "en", max_chars: int = 1200):
        self.lang = lang
        self.max_chars = max_chars

    def _lazy_import(self):
        global wikipedia
        import wikipedia  # type: ignore
        wikipedia.set_lang(self.lang)

    def search_and_summarize(self, query: str, n: int = 3) -> List[Dict[str, str]]:
        try:
            self._lazy_import()
            titles = wikipedia.search(query)[:n]
            results = []
            for t in titles:
                try:
                    page = wikipedia.page(t, auto_suggest=False)
                    summ = wikipedia.summary(t, sentences=6)
                    results.append({
                        "source": f"wikipedia:{page.title}",
                        "url": page.url,
                        "text": summ[: self.max_chars]
                    })
                except Exception:
                    continue
            return results
        except Exception:
            return []

# ---- Tool 3: LLM for synthesis (FLAN-T5 on CPU)
class LLMTool:
    def __init__(self, model_name: str = "google/flan-t5-base", max_new_tokens: int = 256):
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.pipe = None

    def _lazy_import(self):
        if self.pipe is None:
            from transformers import pipeline  # type: ignore
            self.pipe = pipeline("text2text-generation", model=self.model_name)

    def answer(self, query: str, evidence: List[Dict[str, str]]) -> str:
        self._lazy_import()
        chunks = []
        for i, ev in enumerate(evidence, 1):
            src = ev.get("source", "unknown")
            txt = ev.get("text", "")
            chunks.append(f"[{i}] {src} :: {txt}")
        context = "\n".join(chunks)
        prompt = (
            "You are an IR agent. Synthesize a concise answer (120-200 words) using the evidence below. "
            "Cite sources in-line like [1], [2]. If evidence is insufficient, say what is missing.\n\n"
            f"Question: {query}\n\nEvidence:\n{context}\n\nAnswer:"
        )
        out = self.pipe(prompt, max_new_tokens=self.max_new_tokens)[0]["generated_text"]
        return out.strip()

# ---- Agent
@dataclass
class AgentConfig:
    max_steps: int = 3
    use_wikipedia: bool = True
    use_local: bool = True

@dataclass
class AgentState:
    query: str
    steps: List[Dict[str, Any]] = field(default_factory=list)
    evidence: List[Dict[str, str]] = field(default_factory=list)
    final_answer: Optional[str] = None

class AgenticIRAgent:
    def __init__(self, docs_dir: str, memory_path: str):
        self.local_tool = LocalCorpusTool(docs_dir, index_path=os.path.join(docs_dir, "faiss.index"))
        self.wiki_tool = WikipediaTool()
        self.llm_tool = LLMTool()
        self.memory_path = memory_path
        self.memory = self._load_memory()

    def _load_memory(self) -> Dict[str, Any]:
        if os.path.exists(self.memory_path):
            try:
                return json.load(open(self.memory_path, "r", encoding="utf-8"))
            except Exception:
                return {}
        return {}

    def _save_memory(self):
        try:
            json.dump(self.memory, open(self.memory_path, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
        except Exception:
            pass

    # This is the new, smarter plan method
    def plan(self, state: AgentState) -> str:
        # Check which sources have been used based on the evidence
        used_sources = {e.get("source", "").split(':')[0] for e in state.evidence}

        # Priority 1: If we have evidence from multiple sources, it's time to synthesize.
        if len(used_sources) > 1 and state.evidence:
            return "SYNTHESIZE"

        # Priority 2: If we only have local evidence, the next logical step is to try Wikipedia.
        if 'local' in used_sources and 'wikipedia' not in used_sources:
            return "TRY_WIKIPEDIA"

        # Priority 3: If we only have Wikipedia evidence, try the local corpus next.
        if 'wikipedia' in used_sources and 'local' not in used_sources:
            return "TRY_LOCAL"

        # Priority 4: If we have no evidence at all, start with a tool. Default to local.
        if not state.evidence:
            return "TRY_LOCAL"

        # Default Fallback: If none of the above, synthesize with what we have.
        return "SYNTHESIZE"

    def act(self, action: str, state: AgentState) -> AgentState:
        if action == "TRY_LOCAL":
            docs = self.local_tool.search(state.query, k=3)
            state.steps.append({"action": action, "num_docs": len(docs)})
            state.evidence.extend(docs)
            self.memory["last_sources"] = "local"
        elif action == "TRY_WIKIPEDIA":
            docs = self.wiki_tool.search_and_summarize(state.query, n=3)
            state.steps.append({"action": action, "num_docs": len(docs)})
            state.evidence.extend(docs)
            self.memory["last_sources"] = "wikipedia"
        elif action == "SYNTHESIZE":
            answer = self.llm_tool.answer(state.query, state.evidence)
            state.steps.append({"action": action, "used_docs": len(state.evidence)})
            state.final_answer = answer
        else:
            state.steps.append({"action": "NOOP"})
        return state

    # This is the new code for Cell 5
    def reflect(self, state: AgentState) -> bool:
        """Return True if we should continue iterating."""
        # NEW RULE: If the answer is too short (<50 words), it's not good enough. Keep trying.
        if state.final_answer and len(state.final_answer.split()) < 50:
            print("--- Reflection: The generated answer is too short. The agent will try to gather more evidence. ---")
            state.final_answer = None  # Erase the bad answer

            # Check if there are other tools to try
            sources = {e.get("source","").split(':')[0] for e in state.evidence}
            if 'local' in sources and 'wikipedia' in sources:
                return False # We've already used all tools, so stop.

            # If we haven't used all tools, continue running.
            return True

        # Original stopping condition: if we have a final answer (that passed the length check), stop.
        if state.final_answer:
            return False

        # Original continuation conditions
        if not state.evidence:
            return True

        sources = [e.get("source","") for e in state.evidence]
        has_local = any(s.startswith("local:") for s in sources)
        has_wiki = any(s.startswith("wikipedia:") for s in sources)
        if not (has_local and has_wiki) and len(state.steps) < 2:
            return True

        return True

    def run(self, query: str, cfg: AgentConfig) -> AgentState:
        state = AgentState(query=query)
        for step in range(cfg.max_steps):
            action = self.plan(state)
            state = self.act(action, state)
            if not self.reflect(state):
                break
        if state.final_answer is None and state.evidence:
            state = self.act("SYNTHESIZE", state)
        self._save_memory()
        return state

print("Agent and Tool classes defined successfully.")

Agent and Tool classes defined successfully.


In [25]:

QUERY = "Impact of climate change on agriculture in India"

DOCS_DIR = "sample_corpus"
MEMORY_PATH = "memory.json"
MAX_STEPS = 3
# --- End of Configuration ---


# Initialize and run the agent
agent = AgenticIRAgent(docs_dir=DOCS_DIR, memory_path=MEMORY_PATH)
cfg = AgentConfig(max_steps=MAX_STEPS)
t0 = time.time()
state = agent.run(QUERY, cfg)
dt = time.time() - t0

# Print the results
print("="*80)
print(f"Query: {QUERY}")
print("\n- Steps -")
for s in state.steps:
    print(s)
print("\n- Evidence -")
for i, ev in enumerate(state.evidence, 1):
    src = ev.get("source","")
    url = ev.get("url","")
    print(f"[{i}] {src} {('('+url+')') if url else ''}")
print("\n- Final Answer -")
print(state.final_answer or "(No answer generated)")
print("-"*80)
print(f"Completed in {dt:.2f}s")
print("="*80)

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (901 > 512). Running this sequence through the model will result in indexing errors


--- Reflection: The generated answer is too short. The agent will try to gather more evidence. ---
Query: Impact of climate change on agriculture in India

- Steps -
{'action': 'TRY_LOCAL', 'num_docs': 3}
{'action': 'TRY_WIKIPEDIA', 'num_docs': 3}
{'action': 'SYNTHESIZE', 'used_docs': 6}
{'action': 'SYNTHESIZE', 'used_docs': 6}

- Evidence -
[1] local:solar_power.txt 
[2] local:llms_intro.txt 
[3] local:rag_explained.txt 
[4] wikipedia:Effects of climate change on agriculture (https://en.wikipedia.org/wiki/Effects_of_climate_change_on_agriculture)
[5] wikipedia:Climate change (https://en.wikipedia.org/wiki/Climate_change)
[6] wikipedia:Climate change in India (https://en.wikipedia.org/wiki/Climate_change_in_India)

- Final Answer -
[4]
--------------------------------------------------------------------------------
Completed in 20.36s


In [24]:
QUERY = "What is solar power"


DOCS_DIR = "sample_corpus"
MEMORY_PATH = "memory.json"
MAX_STEPS = 3
# --- End of Configuration ---


# Initialize and run the agent
agent = AgenticIRAgent(docs_dir=DOCS_DIR, memory_path=MEMORY_PATH)
cfg = AgentConfig(max_steps=MAX_STEPS)
t0 = time.time()
state = agent.run(QUERY, cfg)
dt = time.time() - t0

# Print the results
print("="*80)
print(f"Query: {QUERY}")
print("\n- Steps -")
for s in state.steps:
    print(s)
print("\n- Evidence -")
for i, ev in enumerate(state.evidence, 1):
    src = ev.get("source","")
    url = ev.get("url","")
    print(f"[{i}] {src} {('('+url+')') if url else ''}")
print("\n- Final Answer -")
print(state.final_answer or "(No answer generated)")
print("-"*80)
print(f"Completed in {dt:.2f}s")
print("="*80)

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (1011 > 512). Running this sequence through the model will result in indexing errors


--- Reflection: The generated answer is too short. The agent will try to gather more evidence. ---
Query: What is solar power

- Steps -
{'action': 'TRY_LOCAL', 'num_docs': 3}
{'action': 'TRY_WIKIPEDIA', 'num_docs': 3}
{'action': 'SYNTHESIZE', 'used_docs': 6}
{'action': 'SYNTHESIZE', 'used_docs': 6}

- Evidence -
[1] local:solar_power.txt 
[2] local:llms_intro.txt 
[3] local:rag_explained.txt 
[4] wikipedia:Solar power by country (https://en.wikipedia.org/wiki/Solar_power_by_country)
[5] wikipedia:Ivanpah Solar Power Facility (https://en.wikipedia.org/wiki/Ivanpah_Solar_Power_Facility)
[6] wikipedia:Concentrated solar power (https://en.wikipedia.org/wiki/Concentrated_solar_power)

- Final Answer -
conversion of energy from sunlight into electricity
--------------------------------------------------------------------------------
Completed in 27.04s


In [26]:
QUERY = "what is genshin impact"

DOCS_DIR = "sample_corpus"
MEMORY_PATH = "memory.json"
MAX_STEPS = 3
# --- End of Configuration ---


# Initialize and run the agent
agent = AgenticIRAgent(docs_dir=DOCS_DIR, memory_path=MEMORY_PATH)
cfg = AgentConfig(max_steps=MAX_STEPS)
t0 = time.time()
state = agent.run(QUERY, cfg)
dt = time.time() - t0

# Print the results
print("="*80)
print(f"Query: {QUERY}")
print("\n- Steps -")
for s in state.steps:
    print(s)
print("\n- Evidence -")
for i, ev in enumerate(state.evidence, 1):
    src = ev.get("source","")
    url = ev.get("url","")
    print(f"[{i}] {src} {('('+url+')') if url else ''}")
print("\n- Final Answer -")
print(state.final_answer or "(No answer generated)")
print("-"*80)
print(f"Completed in {dt:.2f}s")
print("="*80)

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (878 > 512). Running this sequence through the model will result in indexing errors


--- Reflection: The generated answer is too short. The agent will try to gather more evidence. ---
Query: what is genshin impact

- Steps -
{'action': 'TRY_LOCAL', 'num_docs': 3}
{'action': 'TRY_WIKIPEDIA', 'num_docs': 2}
{'action': 'SYNTHESIZE', 'used_docs': 5}
{'action': 'SYNTHESIZE', 'used_docs': 5}

- Evidence -
[1] local:rag_explained.txt 
[2] local:llms_intro.txt 
[3] local:solar_power.txt 
[4] wikipedia:Paimon (Genshin Impact) (https://en.wikipedia.org/wiki/Paimon_(Genshin_Impact))
[5] wikipedia:Furina (Genshin Impact) (https://en.wikipedia.org/wiki/Furina_(Genshin_Impact))

- Final Answer -
a 2020 action role-playing gacha game
--------------------------------------------------------------------------------
Completed in 28.45s


## Reflection Questions and Answers

####**Q1. How does an agentic approach improve over a static IR pipeline?**

>An agentic approach, like the one implemented here, improves over a static IR pipeline by allowing for dynamic interaction and adaptation. Instead of a fixed sequence of steps, the agent can reason about the current state (e.g., what evidence has been gathered) and decide on the next best action (e.g., try a different tool, synthesize the current evidence). This allows the agent to potentially gather more relevant information and produce a better answer, especially for complex queries where a single search might not be sufficient. The reflection step further enhances this by allowing the agent to evaluate the quality of the generated answer and decide if further steps are needed.

####**Q2. What are the limitations of using Wikipedia and a small local corpus as tools?**

>The limitations of using Wikipedia and a small local corpus as tools are primarily related to the scope and depth of information available.

>*   **Wikipedia:** While a vast resource, Wikipedia might not have detailed or specialized information on every topic. It can also be subject to biases or inaccuracies, and its content might not always be up-to-date.
>*   **Small Local Corpus:** A small local corpus is limited to the specific documents it contains. If the query is outside the domain of the local documents, this tool will be ineffective. It also requires manual effort to curate and maintain.

>Both sources might lack the real-time information or specific domain expertise needed for certain queries.

####**Q3. How would you extend this agent to work in a real-world scenario (e.g., academic search engines)?**

>To extend this agent to work in a real-world academic search scenario, several enhancements would be needed:

>*   **Tool Integration:** Integrate with academic search APIs (e.g., Semantic Scholar, PubMed, Google Scholar) to access peer-reviewed articles, conference papers, and other scholarly resources.
>*   **Improved Parsing and Chunking:** Develop more sophisticated methods for parsing academic papers, which often have complex structures (sections, figures, tables, references). This would involve intelligent chunking to preserve the context within documents.
>*   **Citation Handling:** Implement robust citation extraction and management to accurately attribute information to its source and potentially build a knowledge graph of related papers.
>*   **Domain-Specific Models:** Consider using or fine-tuning language models that are specialized for academic text, which often contains technical jargon and complex sentence structures.
>*   **Advanced Reflection:** Enhance the reflection mechanism to evaluate the relevance and credibility of academic sources, potentially using metrics like citation counts or journal impact factors.
>*   **Interactive Refinement:** Allow for user interaction to guide the agent's search and synthesis process, enabling users to provide feedback on relevance or request deeper dives into specific aspects of a topic.
>*   **Scalable Indexing:** For a large-scale academic corpus, use more scalable indexing solutions beyond a simple FAISS index on a single machine.